# Step 3: Encoder tests
`Encoder` (`CharEmbedder` → `BiLSTM` → `MaskedPooling`) lives in `08_encoder_step.py`. The stage notebooks (05–07) cover each stage's details; this one tests the composition end to end.

Tests to write: shape `(B, 256)`; total parameters 167,168; **padding test** (same address padded to T = 10 vs T = 50, `eval()` + `no_grad()`, `allclose`, both `pooling="mean"` and `"max"`); real-data smoke test (32 `ot_train` addresses, finite output).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import torch
from torch import nn

from helpers.pairs import load_pairs

_vocab_step = import_module("siamese_model.01_vocab_step")       # digit-prefixed filenames -> can't use `from ... import`
build_vocab = _vocab_step.build_vocab
PAD_ID, UNK_ID = _vocab_step.PAD_ID, _vocab_step.UNK_ID
encode_batch = import_module("siamese_model.03_encode_batch_step").encode_batch
Encoder = import_module("siamese_model.08_encoder_step").Encoder

In [2]:
ot_train, ot_train_label = load_pairs("overture", "train")   # only the train split: the vocab is built from train
vocab = build_vocab(ot_train)
vocab

{'<pad>': 0,
 '<unk>': 1,
 ' ': 2,
 'e': 3,
 't': 4,
 'a': 5,
 's': 6,
 'r': 7,
 'n': 8,
 'o': 9,
 'i': 10,
 'd': 11,
 '1': 12,
 'l': 13,
 '2': 14,
 '0': 15,
 'm': 16,
 '5': 17,
 'v': 18,
 '3': 19,
 'w': 20,
 '4': 21,
 'h': 22,
 'p': 23,
 '6': 24,
 'c': 25,
 '7': 26,
 '9': 27,
 '8': 28,
 'b': 29,
 'u': 30,
 'g': 31,
 'k': 32,
 'y': 33,
 'f': 34,
 'x': 35,
 'q': 36,
 '-': 37,
 'j': 38,
 'z': 39}

## Tests

In [16]:
torch.manual_seed(0)
strings = [
    "a", "aaaa"
]
x, lengths = encode_batch(strings, vocab)

# Check padding and sanity check
for mode in ("mean", "max"):
    enc = Encoder(len(vocab), pooling=mode).eval()
    ids_a, len_a = encode_batch(["12 main st", "x" * 10], vocab)
    ids_b, len_b = encode_batch(["12 main st", "x" * 50], vocab)
    assert ids_a.shape[1] == 10 and ids_b.shape[1] == 50   # the padding really differs
    with torch.no_grad():
        assert torch.allclose(enc(ids_a, len_a)[0], enc(ids_b, len_b)[0], atol=1e-6)
# Check that the output length makes sense and is same for mean and mode
for mode in ("mean", "max"):
    encoder = Encoder(len(vocab), pooling = mode)
    output = encoder(x, lengths)
    assert output.shape == (2, 256)

# Check params size
encoder = Encoder(len(vocab))
params = encoder.named_parameters()
assert sum(p.numel() for p in encoder.parameters()) == 167168
output = encoder(x,lengths).sum().backward()
assert not any(
    p.grad.abs().sum() == 0 for name,p in params
)

# Smoke test
real_samples = ot_train["address_a"].head(32).to_list()
ids_a, len_a = encode_batch(real_samples, vocab)
enc = Encoder(len(vocab))
out = enc(ids_a, len_a)
assert out.shape == (32, 256)
assert torch.isfinite(out).all()



